# 1. Data Collection and ETF Universe Definition

## Overview
This notebook covers the first step in creating a DIY ETF portfolio: collecting and organizing ETF data. We'll:
1. Define our investment universe based on GDP and MSCI classification
2. Scrape ETF data from JustETF
3. Verify ETF availability on our chosen platform

## Required Libraries and Setup

In [8]:
#!pip uninstall justetf-scraping
!pip install -e "G:\My Drive\dev\justetf-scraping"
!pip list

Obtaining file:///G:/My%20Drive/dev/justetf-scraping
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for justetf-scraping (pyproject.toml): started
  Building editable for justetf-scraping (pyproject.toml): finished with status 'done'
  Created wheel for justetf-scraping: filename=justetf_scraping-0.1.0-py3-none-any.whl size=5260 sha256=7d08feda94798e1d759329ebb06893b3df269b0fb2774415edc9014c4d023eda
  Stored in directory: C:\Users\rakes\AppData\Local\Temp\pip-ephem-wheel-cache-94y6oojl\wheels\2e\f6\f5\bf89eaac5a2832a794abd18b

In [2]:
import pandas as pd
import numpy as np
import requests
import json
import justetf_scraping
import os

# Install required package if not already installed
#!pip install git+https://github.com/druzsan/justetf-scraping.git


justETF scraping module loaded.


## Define Investment Universe
We'll start by creating a DataFrame of major economies, including their:
- GDP (2023 estimates)
- MSCI market classification (Developed/Emerging)
- Geographic region
- Country code for data collection

In [3]:
# Create the country DataFrame using list of lists


country_data = [
    ["United States", "$27.721 trillion", "Developed", "AmericasandUK", "US", "USD"],
    ["Canada", "$2.142 trillion", "Developed", "AmericasandUK", "CA", "CAD"],
    ["United Kingdom", "$3.381 trillion", "Developed", "AmericasandUK", "GB", "GBP"],
    ["Japan", "$4.204 trillion", "Developed", "APAC", "JP", "JPY"],
    ["Australia", "$1.728 trillion", "Developed", "APAC", "AU", "AUD"],
    ["Germany", "$4.526 trillion", "Developed", "EMEA", "DE", "EUR"],
    ["France", "$3.052 trillion", "Developed", "EMEA", "FR", "EUR"],
    ["Italy", "$2.301 trillion", "Developed", "EMEA", "IT", "EUR"],
    ["Spain", "$1.620 trillion", "Developed", "EMEA", "ES", "EUR"],
    ["Netherlands", "$1.154 trillion", "Developed", "EMEA", "NL", "EUR"],
    ["Switzerland", "$884.94 billion", "Developed", "EMEA", "CH", "CHF"],
    ["Brazil", "$2.174 trillion", "Emerging", "Americas", "BR", "BRL"],
    ["Mexico", "$1.789 trillion", "Emerging", "Americas", "MX", "MXN"],
    ["China", "$17.795 trillion", "Emerging", "APACandEMEA", "CN", "CNY"],
    ["India", "$3.568 trillion", "Emerging", "APACandEMEA", "IN", "INR"],
    ["South Korea", "$1.713 trillion", "Emerging", "APACandEMEA", "KR", "KRW"],
    ["Indonesia", "$1.371 trillion", "Emerging", "APACandEMEA", "ID", "IDR"]
]

#Add currency to the each country


columns = ["Country", "2023 GDP", "MSCI", "Region", "Short_name", "Currency"]
country_df = pd.DataFrame(country_data, columns=columns)


#sort by MSCI then region
country_df.sort_values(by=["MSCI", "Region"], inplace=True)

## ETF Data Collection
Now we'll collect ETF data for both equity and bond ETFs across our defined regions:

In [4]:
import traceback
# Define asset classes to collect
asset_classes = [
    "class-equity", 
    "class-bonds"
]

# Group countries by region for data organization
region_countries = country_df.groupby('Region')['Short_name'].apply(list).to_dict()

# Create a reverse mapping of country to region
country_to_region = dict(zip(country_df['Short_name'], country_df['Region']))
country_to_country_name = dict(zip(country_df['Short_name'], country_df['Country']))
country_to_msci = dict(zip(country_df['Short_name'], country_df['MSCI']))

# Create a mapping of MSCI and Region to Category
msci_region_to_category = {
    ('Developed', 'AmericasandUK'): 'Developed_AmericasandUK',
    ('Developed', 'EMEA'): 'Developed_EMEA',
    ('Developed', 'APAC'): 'Developed_APAC',
    ('Emerging', 'Americas'): 'Emerging_Americas',
    ('Emerging', 'APACandEMEA'): 'Emerging_APACandEMEA'
}

# Iterate over asset classes and countries
for asset_class in asset_classes:
    for country in country_df['Short_name']:
        try:
            # Load ETF data by country if equity, else load by currency
            if asset_class == "class-equity":
                df = justetf_scraping.load_overview(asset_class=asset_class, country=country, local_country="GB")
            else:
                # For bonds, we need to specify the currency as well
                currency = country_df[country_df['Short_name'] == country]['Currency'].values[0]
                df = justetf_scraping.load_overview(asset_class=asset_class, instrument_currency=currency, local_country="GB")
                
                        
            # Add region and country information
            df['region'] = country_to_region[country]
            df['country'] = country_to_country_name[country]
            
            # Map MSCI and Region to Category
            msci = country_to_msci[country]
            region = country_to_region[country]
            region_category = msci_region_to_category.get((msci, region), "Unknown")
            df['region_category'] = region_category
            
            # Create filename based on asset class and category
            filename = f'justetf_{asset_class}_{region_category.lower()}.csv'.lower()
            
            # Append to existing file if it exists, otherwise create new
            if os.path.exists(filename):
                existing_df = pd.read_csv(filename)
                combined_df = pd.concat([existing_df, df], ignore_index=True).drop_duplicates(subset=['ticker'])
                combined_df.to_csv(filename, index=False)
            else:
                df.to_csv(filename, index=False)
            
            print(f"Processed {country} {currency} data for {asset_class}")
        except Exception as e:
            print(f"Error scraping {asset_class} for {country}: {e}")
            #traceback.print_exc()

loading overview
getting raw overview
building etf_params
building etf_params
building etf_params


G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")
G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")


Error scraping class-equity for JP: name 'currency' is not defined
loading overview
getting raw overview
building etf_params
building etf_params
building etf_params


G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")
G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")


Error scraping class-equity for AU: name 'currency' is not defined
loading overview
getting raw overview
building etf_params
building etf_params
building etf_params


...

Processed CH CHF data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search


G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")
G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")


Processed CN CNY data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search


G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")
G:\My Drive\dev\justetf-scraping\justetf_scraping\overview.py:420: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column] = df[column].replace({"Yes": True, "No": False}).astype("bool")


Processed IN INR data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search
Processed KR KRW data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search
Processed ID IDR data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search
Processed BR BRL data for class-bonds
loading overview
getting raw overview
building etf_params
adding currency to search
building etf_params
adding currency to search
building etf_params
adding currency to search
Processed MX MXN data for class-bonds


In [5]:
country_to_country_name

{'JP': 'Japan',
 'AU': 'Australia',
 'US': 'United States',
 'CA': 'Canada',
 'GB': 'United Kingdom',
 'DE': 'Germany',
 'FR': 'France',
 'IT': 'Italy',
 'ES': 'Spain',
 'NL': 'Netherlands',
 'CH': 'Switzerland',
 'CN': 'China',
 'IN': 'India',
 'KR': 'South Korea',
 'ID': 'Indonesia',
 'BR': 'Brazil',
 'MX': 'Mexico'}

## Data Collection Summary
Display summary of collected ETFs by region:

In [6]:
asset_classes, country_df

(['class-equity', 'class-bonds'],
            Country          2023 GDP       MSCI         Region Short_name  \
 3            Japan   $4.204 trillion  Developed           APAC         JP   
 4        Australia   $1.728 trillion  Developed           APAC         AU   
 0    United States  $27.721 trillion  Developed  AmericasandUK         US   
 1           Canada   $2.142 trillion  Developed  AmericasandUK         CA   
 2   United Kingdom   $3.381 trillion  Developed  AmericasandUK         GB   
 5          Germany   $4.526 trillion  Developed           EMEA         DE   
 6           France   $3.052 trillion  Developed           EMEA         FR   
 7            Italy   $2.301 trillion  Developed           EMEA         IT   
 8            Spain   $1.620 trillion  Developed           EMEA         ES   
 9      Netherlands   $1.154 trillion  Developed           EMEA         NL   
 10     Switzerland   $884.94 billion  Developed           EMEA         CH   
 13           China  $17.795 t

In [ ]:
# Summarize collected data
summary_data = []
#iterate over csvs
for csvl in os.listdir('.'):
    if csvl.startswith('justetf_class-') and csvl.endswith('.csv'):        

for asset_class in asset_classes:
    for msci_category in regional_risk_weights['Category'].unique():
        filename = f'justetf_{asset_class}_{region_category.lower()}.csv'.lower()
        print(filename)
        if os.path.exists(filename):
            df = pd.read_csv(filename)
            summary_data.append({
                'Asset Class': asset_class.replace('class-', '').title(),
                'Category': msci_category,
                'NumberofETFs': len(df),
                'DistributingETFs': len(df[df['dividends'] == 'Distributing'])
            })

summary_df = pd.DataFrame(summary_data)
display(summary_df.pivot_table(
    index='Category',
    columns='Asset Class',
    values=['NumberofETFs', 'DistributingETFs'],
    aggfunc='first'
))

justetf_class-equity_emerging_americas.csv
justetf_class-equity_emerging_americas.csv
justetf_class-equity_emerging_americas.csv
justetf_class-equity_emerging_americas.csv
justetf_class-equity_emerging_americas.csv
justetf_class-bonds_emerging_americas.csv
justetf_class-bonds_emerging_americas.csv
justetf_class-bonds_emerging_americas.csv
justetf_class-bonds_emerging_americas.csv
justetf_class-bonds_emerging_americas.csv


DistributingETFs        NumberofETFs       
Asset Class                          Bonds Equity        Bonds Equity
Category                                                             
Developed APAC                           0      1            1      7
Developed Americas and UK                0      1            1      7
Developed EMEA                           0      1            1      7
Emerging APAC and EMEA                   0      1            1      7
Emerging Americas                        0      1            1      7

In [43]:
summary_df

,Asset Class,Region,Number of ETFs,Distributing ETFs
0,Equity,APAC,208,54
1,Equity,Americas,468,141
2,Equity,EMEA,143,66
3,Bonds,APAC,35,14
4,Bonds,Americas,178,94
5,Bonds,EMEA,63,42


In [97]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=ETF_PROFILE&symbol=AUAD.LON&apikey=1JYCUCSEKO7IYQKU'
r = requests.get(url)
data = r.json()
data


{}

In [96]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol=AUAD.LON&outputsize=full&apikey=1JYCUCSEKO7IYQKU'
r = requests.get(url)
data = r.json()
data

df = pd.DataFrame(data['Time Series (Daily)']).T
df.index = pd.to_datetime(df.index)
df['5. adjusted close'] = pd.to_numeric(df['5. adjusted close'], errors='coerce')
df
            

,1. open,2. high,3. low,4. close,5. adjusted close,6. volume,7. dividend amount,8. split coefficient
2025-05-14,1838.5,1843.252,1833.75,1833.75,1833.7500,1381,0.0000,1.0
2025-05-13,1838.5,1853.5,1838.333,1853.5,1853.5000,125,0.0000,1.0
2025-05-12,1845.5,1855.5,1835.0,1835.0,1835.0000,197,0.0000,1.0
2025-05-09,1817.5,1821.5,1817.5,1818.25,1818.2500,3,0.0000,1.0
2025-05-07,1816.066,1816.066,1804.5,1805.75,1805.7500,272,0.0000,1.0
...,...,...,...,...,...,...,...,...
2018-12-24,1472.0,1472.0,1422.0,1422.0,1082.0824,10000,0.0000,1.0
2018-08-21,1667.5,1667.5,1660.5,1660.5,1263.5709,4,0.0000,1.0
2018-05-15,1557.0,1560.25,1557.0,1560.25,1170.4240,6102,0.0000,1.0
2018-03-16,1523.5,1523.5,1523.5,1523.5,1142.8560,19,0.0000,1.0


In [11]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords=DBC&apikey=1JYCUCSEKO7IYQKU'
r = requests.get(url)
data = r.json()

print(data)

{'bestMatches': [{'1. symbol': 'DBC', '2. name': 'Invesco DB Commodity Index Tracking Fund', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '1.0000'}, {'1. symbol': 'DBCCF', '2. name': 'Decibel Cannabis Company Inc', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.7500'}, {'1. symbol': 'DBCMX', '2. name': 'DOUBLELINE STRATEGIC COMMODITY FUND CLASS I', '3. type': 'Mutual Fund', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.7500'}, {'1. symbol': 'DBC-P.TRV', '2. name': 'Departure Bay Capital Corp.', '3. type': 'Equity', '4. region': 'Toronto Venture', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-05', '8. currency': 'CAD',

In [91]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords=C024&apikey=1JYCUCSEKO7IYQKU'
r = requests.get(url)
data = r.json()

print(data)

{'bestMatches': [{'1. symbol': 'C024.FRK', '2. name': 'Lyxor FTSE China A50 UCITS ETF', '3. type': 'ETF', '4. region': 'Frankfurt', '5. marketOpen': '08:00', '6. marketClose': '20:00', '7. timezone': 'UTC+02', '8. currency': 'EUR', '9. matchScore': '0.8000'}, {'1. symbol': 'C024.DEX', '2. name': 'Amundi MSCI China A II ETF EUR', '3. type': 'ETF', '4. region': 'XETRA', '5. marketOpen': '08:00', '6. marketClose': '20:00', '7. timezone': 'UTC+02', '8. currency': 'EUR', '9. matchScore': '0.7273'}]}


In [1]:
#!pip install openbb
import openbb_terminal
from openbb_terminal.sdk import openbb

# Fetch historical data for the ASX index
asx_data = openbb.stocks.load('ASX')

# Display the data
print(asx_data)

ModuleNotFoundError: No module named 'openbb_terminal'

In [7]:
from openbb import obb
obb.index.available(provider="yfinance").to_df()


,name,code,symbol
0,S&P 500 Index,sp500,^GSPC
1,S&P 500 Index,spx,^SPX
2,S&P 400 Mid Cap Index,sp400,^SP400
3,S&P 600 Small Cap Index,sp600,^SP600
4,S&P 500 TR Index,sp500tr,^SP500TR
...,...,...,...
269,CBOE Euro Currency Volatility Index,cboe_evz,^EVZ
270,CBOE Russell 2000 Volatility Index,cboe_rvx,^RVX
271,ICE BofAML Move Index,move,^MOVE
272,US Dollar Index,dxy,DX-Y.NYB


In [9]:
obb.equity.price.historical(symbol="SPY", provider="yfinance").to_df()


1 Failed download:
['SPY']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


EmptyDataError: 
[Empty] -> No results found. Try adjusting the query parameters.

In [ ]:
obb.equity.search("IBZL").to_df()
 

OpenBBError: Results not found.

In [2]:
pip install openbb-terminal

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement openbb-terminal (from versions: none)
ERROR: No matching distribution found for openbb-terminal
